# Cross-Model Activation Transfer


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from matplotlib.lines import Line2D
from scipy.spatial import cKDTree
from umap import UMAP

from activation_cross_analysis import (
    calc_bidirectional_top_k_connections,
    calc_cross_similarity,
    calc_transport_latents,
)
from capture.saving import load_captured_activations


# Setup

Load captured activations once and convert tensors to NumPy matrices with neurons as rows and examples as columns.


In [ ]:
results_path = Path("results")
captured = load_captured_activations(results_path)
tasks = tuple(captured)

activations = {}
for task in tasks:
    activations[task] = {
        "base": captured[task]["base"].states.detach().cpu().numpy().T,
        "finetuned": captured[task]["finetuned"].states.detach().cpu().numpy().T,
    }


# Plot Placeholders

These are intentionally empty until plotting is implemented.


In [ ]:
def sampled_positions(length, max_ticks=10):
    if length <= 0:
        return np.array([], dtype=int)
    tick_count = min(max_ticks, length)
    return np.unique(np.linspace(0, length - 1, tick_count, dtype=int))


def set_neuron_ticks(axis, positions, labels, orientation):
    if orientation == "x":
        axis.set_xticks(positions)
        axis.set_xticklabels(labels, rotation=90)
    else:
        axis.set_yticks(positions)
        axis.set_yticklabels(labels)


def signed_heatmap(axis, matrix, title, x_order=None, y_order=None):
    matrix = np.asarray(matrix, dtype=float)
    max_abs = np.nanmax(np.abs(matrix)) if matrix.size else 1.0
    color_limit = max_abs if np.isfinite(max_abs) and max_abs > 0.0 else 1.0
    image = axis.imshow(
        matrix,
        cmap="coolwarm",
        vmin=-color_limit,
        vmax=color_limit,
        interpolation="nearest",
        aspect="auto",
    )
    axis.set_title(title)
    axis.set_xlabel("fine-tuned neuron")
    axis.set_ylabel("base neuron")

    y_order = np.arange(matrix.shape[0]) if y_order is None else np.asarray(y_order)
    x_order = np.arange(matrix.shape[1]) if x_order is None else np.asarray(x_order)
    y_positions = sampled_positions(matrix.shape[0])
    x_positions = sampled_positions(matrix.shape[1])
    set_neuron_ticks(axis, x_positions, [str(int(x_order[position])) for position in x_positions], "x")
    set_neuron_ticks(axis, y_positions, [str(int(y_order[position])) for position in y_positions], "y")
    return image


def draw_spectral_ordering(task, metric, ordered_W, base_order, finetuned_order):
    figure, axis = plt.subplots(figsize=(10, 8))
    image = signed_heatmap(
        axis,
        ordered_W,
        f"{task} {metric} SVD spectral ordering",
        x_order=finetuned_order,
        y_order=base_order,
    )
    figure.colorbar(image, ax=axis, label=metric)
    figure.tight_layout()
    plt.show()


def draw_singular_values(task, metric, singular_values):
    ranks = np.arange(1, len(singular_values) + 1)
    figure, axis = plt.subplots(figsize=(8, 5))
    axis.plot(ranks, singular_values, marker="o", linewidth=1.5, markersize=3)
    axis.set_title(f"{task} {metric} singular values")
    axis.set_xlabel("rank")
    axis.set_ylabel("singular value")
    axis.grid(True, alpha=0.3)
    figure.tight_layout()
    plt.show()


def project_transport_latents(Pb, Pf):
    latents = np.vstack([Pb, Pf])
    neighbor_count = min(15, max(2, latents.shape[0] - 1))
    projected = UMAP(n_components=2, n_neighbors=neighbor_count, random_state=0).fit_transform(latents)

    return projected[: Pb.shape[0]], projected[Pb.shape[0] :]


def draw_transport_latents(task, metric, base_xy, finetuned_xy, rank, projection_label):
    base_xy = np.asarray(base_xy, dtype=float)
    finetuned_xy = np.asarray(finetuned_xy, dtype=float)

    figure, axis = plt.subplots(figsize=(10, 8))
    axis.scatter(base_xy[:, 0], base_xy[:, 1], color="tab:blue", label="base", s=18, alpha=0.75)
    axis.scatter(finetuned_xy[:, 0], finetuned_xy[:, 1], color="tab:orange", label="fine-tuned", s=18, alpha=0.75)

    for neuron_index, (x, y) in enumerate(base_xy):
        axis.text(x, y, str(neuron_index), fontsize=5, color="tab:blue", alpha=0.65)
    for neuron_index, (x, y) in enumerate(finetuned_xy):
        axis.text(x, y, str(neuron_index), fontsize=5, color="tab:orange", alpha=0.65)

    axis.set_title(f"{task} {metric} rank-{rank} transport latents via {projection_label}")
    axis.set_xlabel(f"{projection_label} dimension 1")
    axis.set_ylabel(f"{projection_label} dimension 2")
    axis.legend()
    axis.grid(True, alpha=0.25)
    figure.tight_layout()
    plt.show()


def draw_top_k_connection_graph(
    task,
    metric,
    query_indices,
    candidate_indices,
    scores,
    k,
    direction,
    relationship,
):
    source_is_base = direction.startswith("base")
    source_prefix = "B" if source_is_base else "F"
    target_prefix = "F" if source_is_base else "B"
    network = nx.DiGraph()
    for position in range(len(scores)):
        source = f"{source_prefix}{int(query_indices[position])}"
        target = f"{target_prefix}{int(candidate_indices[position])}"
        network.add_edge(
            source,
            target,
            score=float(scores[position]),
            magnitude=abs(float(scores[position])),
        )

    if not network.nodes:
        return
    base_nodes = [node for node in network if node.startswith("B")]
    finetuned_nodes = [node for node in network if node.startswith("F")]
    node_count = len(network)
    source_nodes = [node for node in network if node.startswith(source_prefix)]
    target_nodes = [node for node in network if node.startswith(target_prefix)]
    primary_hubs = {}
    hub_spokes = {hub: [] for hub in target_nodes}
    for source in source_nodes:
        outgoing = list(network.out_edges(source, data=True))
        primary_hub = max(outgoing, key=lambda edge: edge[2]["magnitude"])[1]
        primary_hubs[source] = primary_hub
        hub_spokes[primary_hub].append(source)

    local_positions = {}
    orbit_radii = {}
    for hub in target_nodes:
        local_positions[hub] = np.zeros(2)
        spokes = sorted(
            hub_spokes[hub],
            key=lambda source: (-network[source][hub]["magnitude"], source),
        )
        cursor = 0
        ring_index = 1
        outer_radius = 0.3
        while cursor < len(spokes):
            ring_capacity = 8 * ring_index
            ring_spokes = spokes[cursor : cursor + ring_capacity]
            ring_radius = 0.45 + 0.35 * (ring_index - 1)
            hub_phase = (int(hub[1:]) * 0.61803398875 % 1.0) * 2.0 * np.pi
            for slot, source in enumerate(ring_spokes):
                angle = hub_phase + 2.0 * np.pi * slot / len(ring_spokes)
                local_positions[source] = ring_radius * np.array([np.cos(angle), np.sin(angle)])
            cursor += len(ring_spokes)
            outer_radius = ring_radius + 0.3
            ring_index += 1
        orbit_radii[hub] = outer_radius

    hub_network = nx.Graph()
    hub_network.add_nodes_from(target_nodes)
    for source in source_nodes:
        primary_hub = primary_hubs[source]
        primary_magnitude = network[source][primary_hub]["magnitude"]
        for _, secondary_hub, edge in network.out_edges(source, data=True):
            if secondary_hub == primary_hub:
                continue
            relation = np.sqrt(primary_magnitude * edge["magnitude"])
            if hub_network.has_edge(primary_hub, secondary_hub):
                hub_network[primary_hub][secondary_hub]["relation"] += relation
            else:
                hub_network.add_edge(primary_hub, secondary_hub, relation=relation)
    for _, _, edge in hub_network.edges(data=True):
        edge["spring_weight"] = 0.25 + np.log1p(edge["relation"])

    hub_count = len(target_nodes)
    hub_scale = max(2.0, 1.2 * np.sqrt(hub_count))
    hub_positions = nx.spring_layout(
        hub_network,
        weight="spring_weight",
        k=1.0 / np.sqrt(hub_count),
        scale=hub_scale,
        seed=0,
        iterations=300,
        method="energy",
        gravity=1.0,
    )

    hub_order = list(target_nodes)
    hub_centers = np.array([hub_positions[hub] for hub in hub_order])
    hub_radii = np.array([orbit_radii[hub] for hub in hub_order])
    separation_margin = 0.2
    search_distance = 2.0 * hub_radii.max() + separation_margin
    for _ in range(200):
        nearby_hubs = cKDTree(hub_centers).query_pairs(search_distance)
        overlap_count = 0
        for first, second in nearby_hubs:
            delta = hub_centers[first] - hub_centers[second]
            distance = np.linalg.norm(delta)
            required_distance = hub_radii[first] + hub_radii[second] + separation_margin
            if distance >= required_distance:
                continue
            if distance == 0.0:
                angle = (first * 0.61803398875 + second) * 2.0 * np.pi
                delta = np.array([np.cos(angle), np.sin(angle)])
                distance = 1.0
            displacement = 0.5 * (required_distance - distance) * delta / distance
            hub_centers[first] += displacement
            hub_centers[second] -= displacement
            overlap_count += 1
        if overlap_count == 0:
            break

    positions = {}
    for hub, center in zip(hub_order, hub_centers):
        positions[hub] = center
        for source in hub_spokes[hub]:
            positions[source] = center + local_positions[source]

    incoming_degrees = dict(network.in_degree())
    node_sizes = {
        node: 40.0 + 18.0 * np.sqrt(incoming_degrees[node])
        for node in network
    }
    figure_side = min(30.0, max(12.0, 8.0 + np.sqrt(node_count) / 4.0))
    figure, axis = plt.subplots(figsize=(figure_side, figure_side))
    orbit_color = "tab:orange" if target_prefix == "F" else "tab:blue"
    for hub in target_nodes:
        axis.add_patch(
            plt.Circle(positions[hub], orbit_radii[hub], fill=False, color=orbit_color, alpha=0.08, linewidth=0.5)
        )
    nx.draw_networkx_nodes(
        network,
        positions,
        nodelist=base_nodes,
        node_color="tab:blue",
        node_size=[node_sizes[node] for node in base_nodes],
        alpha=0.85,
        ax=axis,
    )
    nx.draw_networkx_nodes(
        network,
        positions,
        nodelist=finetuned_nodes,
        node_color="tab:orange",
        node_size=[node_sizes[node] for node in finetuned_nodes],
        alpha=0.9,
        ax=axis,
    )
    edge_magnitudes = np.array([data["magnitude"] for _, _, data in network.edges(data=True)])
    affinity_magnitudes = np.clip(edge_magnitudes, 0.0, 1.0)
    edge_cmap = plt.get_cmap("Greens" if relationship == "correlation" else "Purples")
    edge_colors = edge_cmap(affinity_magnitudes)
    edge_colors[:, 3] = 0.05 + 0.9 * affinity_magnitudes ** 2
    edge_widths = 0.25 + 2.75 * affinity_magnitudes ** 2
    arrow_sizes = 4.0 + 8.0 * affinity_magnitudes
    network_nodes = list(network)
    nx.draw_networkx_edges(
        network,
        positions,
        width=edge_widths,
        edge_color=edge_colors,
        arrows=True,
        arrowsize=arrow_sizes.tolist(),
        arrowstyle="-|>",
        node_size=[node_sizes[node] for node in network_nodes],
        nodelist=network_nodes,
        min_source_margin=2,
        min_target_margin=2,
        ax=axis,
    )
    neuron_labels = {node: node[1:] for node in network}
    label_size = 4 if node_count > 500 else 6
    nx.draw_networkx_labels(network, positions, labels=neuron_labels, font_size=label_size, ax=axis)
    axis.set_title(f"{task} {metric} {relationship} {direction} top-{k} connections")
    axis.legend(
        handles=[
            Line2D([], [], marker="o", linestyle="None", color="tab:blue", markersize=7, label="base"),
            Line2D([], [], marker="o", linestyle="None", color="tab:orange", markersize=7, label="fine-tuned"),
        ]
    )
    affinity_scale = plt.cm.ScalarMappable(cmap=edge_cmap, norm=plt.Normalize(0.0, 1.0))
    figure.colorbar(affinity_scale, ax=axis, fraction=0.03, pad=0.01, label=f"absolute {metric} affinity")
    axis.text(0.01, 0.01, "node area scales with incoming top-k edges", transform=axis.transAxes, fontsize=8)
    axis.axis("off")
    figure.tight_layout()
    plt.show()
    return network, positions, primary_hubs, orbit_radii


### Cross Similarity

Build the shared cross-model similarity source W for downstream cells.


In [ ]:
similarity_by_task_metric = {}
base_self_similarity_by_task_metric = {}
cross_minus_base_self_by_task_metric = {}
for task in tasks:
    base_states = activations[task]["base"]
    finetuned_states = activations[task]["finetuned"]
    cross_pearson, cross_cosine = calc_cross_similarity(base_states, finetuned_states)
    base_pearson, base_cosine = calc_cross_similarity(base_states, base_states)
    for metric, cross_similarity, base_self_similarity in (
        ("pearson", cross_pearson, base_pearson),
        ("cosine", cross_cosine, base_cosine),
    ):
        similarity_by_task_metric[task, metric] = cross_similarity
        base_self_similarity_by_task_metric[task, metric] = base_self_similarity
        cross_minus_base_self_by_task_metric[task, metric] = (
            cross_similarity - base_self_similarity
        )


### Cross Minus Base-Self Similarity

For base neuron $i$ and fine-tuned neuron $j$, compare the observed cross-model affinity $R_{XY}[i,j]$ with the base-model relationship that already existed between $i$ and $j$, $R_{XX}[i,j]$. The signed matrix $R_{XY} - R_{XX}$ shows newly strengthened (positive) and weakened (negative) pairwise affinities.


In [ ]:
for (task, metric), cross_minus_base_self in cross_minus_base_self_by_task_metric.items():
    figure, axis = plt.subplots(figsize=(10, 8))
    image = signed_heatmap(
        axis,
        cross_minus_base_self,
        f"{task} {metric} base-to-fine-tuned minus base-self similarity",
    )
    figure.colorbar(image, ax=axis, label=f"{metric} similarity change")
    figure.tight_layout()
    plt.show()


# Cross-Distribution Visualizer


In [ ]:
distribution_bin_count = 128
distribution_z_bound = 1.0
distribution_bin_edges = np.linspace(-1.0, 1.0, distribution_bin_count + 1)

for (task, metric), W in similarity_by_task_metric.items():
    distribution_frequencies = np.zeros((W.shape[0], distribution_bin_count), dtype=int)
    distribution_means = np.full(W.shape[0], np.nan, dtype=float)
    distribution_standard_deviations = np.full(W.shape[0], np.nan, dtype=float)

    for base_neuron, similarities in enumerate(W):
        finite_similarities = similarities[np.isfinite(similarities)]
        if finite_similarities.size == 0:
            continue
        finite_similarities = np.clip(finite_similarities, -1.0, 1.0)
        distribution_frequencies[base_neuron], _ = np.histogram(
            finite_similarities,
            bins=distribution_bin_edges,
        )
        distribution_means[base_neuron] = finite_similarities.mean()
        distribution_standard_deviations[base_neuron] = finite_similarities.std()

    lower_z_bound = np.clip(
        distribution_means - distribution_z_bound * distribution_standard_deviations,
        -1.0,
        1.0,
    )
    upper_z_bound = np.clip(
        distribution_means + distribution_z_bound * distribution_standard_deviations,
        -1.0,
        1.0,
    )
    base_neurons = np.arange(W.shape[0])

    figure, axis = plt.subplots(figsize=(12, 10))
    image = axis.imshow(
        distribution_frequencies,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        extent=(-1.0, 1.0, -0.5, W.shape[0] - 0.5),
        cmap="magma",
    )
    axis.plot(distribution_means, base_neurons, color="cyan", linewidth=1.0, label="mean")
    axis.plot(
        lower_z_bound,
        base_neurons,
        color="white",
        linewidth=0.8,
        linestyle="--",
        label=f"z = ±{distribution_z_bound:g}",
    )
    axis.plot(upper_z_bound, base_neurons, color="white", linewidth=0.8, linestyle="--")
    axis.set_title(f"{task} {metric} base-to-fine-tuned similarity distributions")
    axis.set_xlabel(f"{metric} similarity")
    axis.set_ylabel("base neuron")
    axis.set_xlim(-1.0, 1.0)
    axis.set_xticks(np.linspace(-1.0, 1.0, 9))
    neuron_ticks = sampled_positions(W.shape[0])
    axis.set_yticks(neuron_ticks)
    axis.set_yticklabels([str(int(neuron)) for neuron in neuron_ticks])
    axis.legend(loc="upper right")
    figure.colorbar(image, ax=axis, label="fine-tuned neuron frequency")
    figure.tight_layout()
    plt.show()


### Shared SVD Artifacts

Compute the SVD of each W exactly once. Singular values, residuals, and low-rank transport cells reuse these artifacts.


In [ ]:
svd_by_task_metric = {}
for (task, metric), W in similarity_by_task_metric.items():
    svd_by_task_metric[task, metric] = np.linalg.svd(W, full_matrices=False)


# CKA

Compute one CKA score per task directly from X and Y, then draw it.


In [ ]:
for task in tasks:
    X = activations[task]["base"]
    Y = activations[task]["finetuned"]

    X_centered = X - X.mean(axis=1, keepdims=True)
    Y_centered = Y - Y.mean(axis=1, keepdims=True)

    numerator = np.linalg.norm(np.matmul(X_centered, Y_centered.T), ord="fro") ** 2
    denominator = np.linalg.norm(np.matmul(X_centered, X_centered.T), ord="fro") * np.linalg.norm(
        np.matmul(Y_centered, Y_centered.T),
        ord="fro",
    )
    cka_score = np.nan if denominator == 0.0 else numerator / denominator

    print(task,"cka:",cka_score)

# Spectral Ordering

Compute the spectral ordering mathematics from W and draw the reordered signed matrix. This SVD is on the normalized affinity used for ordering, not the reusable SVD of W.


In [ ]:
for (task, metric), W in similarity_by_task_metric.items():
    affinity = np.abs(W)
    row_degree = affinity.sum(axis=1)
    column_degree = affinity.sum(axis=0)
    normalized = np.divide(
        affinity,
        np.sqrt(row_degree[:, None] * column_degree[None, :]),
        out=np.zeros_like(affinity, dtype=float),
        where=(row_degree[:, None] > 0.0) & (column_degree[None, :] > 0.0),
    )

    spectral_U, spectral_s, spectral_Vt = np.linalg.svd(normalized, full_matrices=False)
    vector_index = 1 if spectral_s.size > 1 else 0
    base_order = np.argsort(spectral_U[:, vector_index], kind="mergesort")
    finetuned_order = np.argsort(spectral_Vt[vector_index], kind="mergesort")
    ordered_W = W[base_order][:, finetuned_order]

    draw_spectral_ordering(task, metric, ordered_W, base_order, finetuned_order)


# Singular Values

Draw singular values from the shared SVD artifacts.


In [ ]:
for (task, metric), (_, singular_values, _) in svd_by_task_metric.items():
    draw_singular_values(task, metric, singular_values)


# Bidirectional Top-K Connection Graphs

Use rank-64 transport latents only for the UMAP projection. Select top-k connections directly from each full-precision similarity matrix and show every selected edge. Sources orbit their strongest selected target in affinity-ordered rings; secondary edges remain visible across orbits. A spring layout places only the hub centers, then separates overlapping orbit envelopes while moving each hub-and-spoke system rigidly.


In [ ]:
transport_rank = 64
connection_k = 5

for (task, metric), W in similarity_by_task_metric.items():
    U, singular_values, Vt = svd_by_task_metric[task, metric]
    k_used = min(transport_rank, singular_values.size)
    Pb, Pf = calc_transport_latents(U, singular_values, Vt, k_used)

    base_xy, finetuned_xy = project_transport_latents(Pb, Pf)
    draw_transport_latents(task, metric, base_xy, finetuned_xy, k_used, "UMAP")

    for relationship in ("correlation", "anticorrelation"):
        base_to_finetuned, finetuned_to_base = calc_bidirectional_top_k_connections(
            W, k=connection_k, relationship=relationship
        )
        base_queries, base_candidates, base_scores, _ = base_to_finetuned
        finetuned_queries, finetuned_candidates, finetuned_scores, _ = finetuned_to_base
        draw_top_k_connection_graph(
            task, metric, base_queries, base_candidates, base_scores,
            connection_k, "base -> fine-tuned", relationship
        )
        draw_top_k_connection_graph(
            task, metric, finetuned_queries, finetuned_candidates, finetuned_scores,
            connection_k, "fine-tuned -> base", relationship
        )
